
# 01 &middot; Validation

You get limited leaderboard submissions today. So the number you'll usually have to assess your model is your **estimate** of your test score.
This notebook is about making a trustworthy estimate!

The one thing to get straight before you start:

> You are not looking for the split that gives you the best score.
> You are looking for the split whose score best matches the test score.

Those pull in opposite directions!

---
### Setup

In [ ]:
#@title Getting things all setup...
# Run me first.
%pip -q install rdkit pandas numpy scipy scikit-learn huggingface_hub fsspec lightgbm matplotlib seaborn
!git clone https://github.com/agura-alt/ai4chem_openadmet.git
%cd ai4chem_openadmet


In [ ]:
#@title Imports...
import os, sys
SETUP_DIR = os.path.abspath("Setup")
os.path.isdir(SETUP_DIR) or sys.exit(f"No Setup dir at {SETUP_DIR}; cwd is {os.getcwd()}")

if SETUP_DIR not in sys.path:
    sys.path.insert(0, SETUP_DIR)

assert os.path.exists("Setup/common.py") and os.path.getsize("Setup/common.py") > 1000, (
    "common.py is missing or truncated. Upload it using the folder icon in the "
    "left sidebar, then re-run this cell.")

sys.modules.pop("common", None)
import common

import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from rdkit import Chem
from rdkit.Chem import Descriptors

sns.set_style("whitegrid")

Change `your-pair-name` to your team name. It has to match the list of registered teams exactly, and be the same in every notebook &mdash; that is what links your work together.

In [ ]:
# Same folder as notebook 00 -- your split, your predictions, your submissions.
common.setup(pair=...)

### 1. Let's fit one model!

Here is the model training loop for one endpoint on one split. A lot of what you do today will look similar to this!

1. Select your molecular representation (may involve descriptor calculation!)
2. Split the rows into train and validation
3. Clean the input data
4. Initialize your model, fit, and predict.

Because we are comparing different validation splits, we will keep the model itself constant.


#### 1.1 Select molecular representation
We're going to use a suite of RDKit descriptors as a quick baseline. Here's how you might implement!

In [ ]:
# 1. select molecular representation
train = common.load_train() # load training data

# molecular representation = rdkit descriptors
smiles = train["SMILES"] # you can always confirm which column is SMILES by looking at train.columns

# making a calculator for all rdkit descriptors
calc = Descriptors.CalcMolDescriptors
desc_names = np.array(Descriptors.descList)[:,0] # just to label our columns!

# assemble descriptors for all SMILES
rows = []
for smi in smiles:
    mol = Chem.MolFromSmiles(smi) if isinstance(smi, str) else None
    if mol is None:
        rows.append({n: np.nan for n in desc_names})
    else:
        try:
            rows.append(calc(mol))
        except Exception:
            rows.append({n: np.nan for n in desc_names})
X_all = pd.DataFrame(rows, columns=desc_names).astype(float)
X_all = X_all.replace([np.inf, -np.inf], np.nan)

print(X_all.shape, "descriptors")
X_all.head()

In [ ]:
# if you ever want to use rdkit descriptors for another model, we provide a convenience function!
X_all = common.rdkit_descriptors(train["SMILES"])

#### 1.2 Training and validation set
This is what the whole notebook is about: picking the right training and validation split. The split will be stored in a variable `fold` which contains a Series of `"train"` and `"val"` labels. We'll split our input data based on that.

In [ ]:
# We'll start with a default split -- you'll implement some more later!
SPLIT = "random"
fold, _ = common.load_split(train, name=SPLIT)

train_idx = (fold == "train").to_numpy() # 1. the molecules we train on
val_idx = (fold == "val").to_numpy()   # 2. the molecules we score on

print(f"{train_idx.sum()} train molecules, {val_idx.sum()} val molecules")

#### 1.3 Clean the input data
Cleaning is a somewhat boring part of model training but there are some choices to make!

We provide a convenience function to perform cleaning, which does the following:
1. Drop all-NaN columns, as well as all constant columns
2. If columns are partially NaNs, median impute the NaNs.

You only want to clean on your training data so you avoid data leakage!

There are other kinds of preprocessing you might want to do -- e.g., dimensionality reduction, feature transformation, etc.

In [ ]:
def clean_features(X_train: pd.DataFrame, *others: pd.DataFrame):
    """Drop all-NaN / constant columns, median-impute the rest.

    Fit on train only -- imputing with statistics from your validation set is
    a quiet way to leak information and flatter your score.
    """
    keep = X_train.columns[(X_train.notna().any()) & (X_train.nunique(dropna=True) > 1)]
    medians = X_train[keep].median()
    out = [X_train[keep].fillna(medians)]
    for other in others:
        out.append(other.reindex(columns=keep).fillna(medians))
    return out[0] if not others else tuple(out)

In [ ]:
X_train, X_val = common.clean_features(X_all[train_idx], X_all[val_idx])
# Note: the first DF is the one that is used to fit the imputer.
# You can add as many additional datasets to be cleaned as you like.

print(f"{X_all.shape[1]} descriptors in -> {X_train.shape[1]} usable\n")
display(X_train.head())          # have a look -- these are the model's inputs

The last cleaning step is to make sure you have an endpoint measurement for all the training and validation data! Here we start with a single endpoint.

In [ ]:
# select endpoint
ENDPOINT = common.ENDPOINTS[0]
print("Chosen endpoint:", ENDPOINT)

y = train.loc[train_idx, ENDPOINT]
ok = y.notna().to_numpy() # selects only the molecules that have an ENDPOINT value
print(f"{ok.sum()} of {len(y)} training molecules have a {ENDPOINT} value")

#### 1.4 Finally, let's train a model!
Our baseline model architecture is the light gradient boosting machine regressor. Steps will involve
- importing the regressor (if you make something custom, you might not do this!)
- initializing the regressor (setting hyperparameters usually)
- fitting the model (this is the training process)
- predicting (generating predictions on the validation set)

In [ ]:
from lightgbm import LGBMRegressor # import
model = LGBMRegressor(n_estimators=400, learning_rate=0.05,
                      num_leaves=31, verbose=-1, n_jobs=-1) # initialize
model.fit(X_train[ok], y[ok]) # train (on only the molecules that have an ENDPOINT value)
y_pred = model.predict(X_val) # predict on the validation set

Now we score!

In [ ]:
# score it: predictions and truth, lined up by molecule
val_preds = pd.DataFrame({"Molecule Name": train.loc[val_idx, "Molecule Name"].to_numpy(),
                          ENDPOINT: y_pred})
truth = train.loc[val_idx].reset_index(drop=True)
display(common.score(truth, val_preds, endpoints=[ENDPOINT]).round(3))

### The same thing, as a function

 Since we are about to run it many times &mdash; nine
endpoints &times; several splits &times; several folds &mdash; all those code is wrapped up
below as `fit_predict_lgbm`.

**When you see `fit_predict_lgbm(fold)` later in this notebook, it is doing exactly
what you just did step-by-step**, once per endpoint, and handing back the predictions
alongside the matching truth rows.


In [ ]:
from lightgbm import LGBMRegressor
from sklearn.base import clone

def fit_predict_lgbm(fold, X=X_all, df=train, endpoints=None):
    """Train on fold=='train', predict fold=='val'. One model per endpoint."""
    endpoints = endpoints or common.ENDPOINTS # select which endpoint you want to predict
    train_idx = (fold == "train").to_numpy()  # extract train/values boolean masks
    val_idx = (fold == "val").to_numpy()
    X_train, X_val = common.clean_features(X[train_idx], X[val_idx]) # remove constant columns, etc.
    out = pd.DataFrame({"Molecule Name": df.loc[val_idx, "Molecule Name"].to_numpy()})

    template = LGBMRegressor(n_estimators=400, learning_rate=0.05,
                             num_leaves=31, verbose=-1, n_jobs=-1)
    for endpoint in endpoints:
        y = df.loc[train_idx, endpoint]
        ok = y.notna().to_numpy()

        ### here is where the model gets trained!        ###
        model = clone(template)   # a fresh, unfitted copy for every endpoint
        model.fit(X_train[ok], y[ok])
        out[endpoint] = model.predict(X_val)
        ### end of model initialize - fit - predict block ###

    return out, df.loc[val_idx].reset_index(drop=True)

In [ ]:
# Same endpoint, same split, same answer -- just one line now.
val_preds, truth = fit_predict_lgbm(fold, endpoints=[ENDPOINT])
display(common.score(truth, val_preds, endpoints=[ENDPOINT]).round(3))

---
## 2. Several splits, one model

You are going to **write these splits yourself**, then run the same model
through each one:

- **random** &mdash; shuffle and cut
- **temporal** &mdash; the last-registered compounds are held out, mimicking the
  real train/test split
- **similarity** &mdash; hold out the molecules least similar to everything else

A split function takes the training frame and returns one label per row:

- **in:** `df`, a DataFrame with `len(df)` rows
- **out:** a `pd.Series` of `"train"` / `"val"`, same length, `index=df.index`

### &#9654;&#65039; Predict first

**Rank the splits from the one that will give the BEST-looking score to the one
that will give the worst. Then say which one you think will be closest to the
real test score.**

*Those two answers should not be the same split. If they are, think again.*

Write your answer here before running the next cell &mdash; one line is enough:

> `your prediction:`

> The runner cell trains one model per endpoint, for every split you
> implemented, so it takes a few minutes. Good time to write down your
> prediction, or to start on the next split.
>
> You don't have to implement them all, unimplemented splits are skipped.
> So feel free to fill them in one at a time!
>
> You do not have to use the scaffolding provided, there are certainly other
> (shorter) ways to implement the splits.

In [ ]:
# Let's start with a random split!
import random
def random_split(df, frac_val=0.2, seed=0):
    random.seed(seed) # set seed for reproducibility
    indices = df.index.tolist() # create a list of indices

    # let's sample from the indices to get the validation set
    ### TODO ###
    # how much do you need to sample?
    val_indices = random.sample(indices, int(...))
    ### END TODO ###

    # let's set the rest as the train
    labels = []
    for i in indices:
        if i in val_indices:
            labels.append("val")
        else:
            labels.append("train")

    return pd.Series(labels, index=df.index)

In [ ]:
# Now a temporal split!
def temporal_split(df, frac_val=0.2):
    # reg_number is just the numeric part of the compound ID, which relates to the time of synthesis
    reg_number = df["Molecule Name"].str.extract(r"(\d+)", expand=False).astype(float)
    sorted_reg =reg_number.sort_values()

    ### TODO ###
    # do you want the first or last fraction of the sorted indices for validation?
    val_indices = sorted_reg.index[...:...]
    ### END TODO ###

    labels = []
    for i in df.index:
        if i in val_indices:
            labels.append("val")
        else:
            labels.append("train")
    return pd.Series(labels, index=df.index)


In [ ]:
# now a chemical similarity split!
# defining chemical similarity is an open question and depends on your molecular descriptors.
# We'll use a simple but effective approach: Morgan fingerprints!
# Feel free to change the radius and size of the fingerprints, or use a different fingerprinting method altogether.

from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator

def similarity_split(df, frac_val=0.2):
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
    fps = []
    for smi in df["SMILES"]:
        mol = Chem.MolFromSmiles(smi)
        fps.append(gen.GetFingerprint(mol) if mol is not None else None)

    # common.py provides a function to compute the nearest neighbour similarity
    # it implements the Tanimoto similarity, which is most suitable for binary fingerprints like Morgan.
    # if you implement a new version with your own chemical descriptors, you might need to change your similarity too!
    sim = common.nearest_neighbour_similarity(fps, fps, exclude_self=True)
    sim_df = pd.DataFrame({"sim": sim})
    sim_df = sim_df.sort_values("sim", ascending=True)

    ### TODO ###
    # do you want the first or last fraction of the sorted indices for validation?
    val_indices = sim_df.index[...:...]
    ### END TODO ###

    labels = []
    for i in df.index:
        if i in val_indices:
            labels.append("val")
        else:
            labels.append("train")
    return pd.Series(labels, index=df.index)

In [ ]:
MY_SPLITS = {"random": random_split,
             "temporal": temporal_split,
             "similarity": similarity_split}

Now we compare the models on those splits!

In [ ]:
results = {}
preds_by_split = {}
MODEL = "lgbm-rdkit"          # the name this model goes into the table under

for name, split_func in MY_SPLITS.items():
    try:
        fold = split_func(train)
    except NotImplementedError:
        print(f"{name:11s} -- not written yet, skipping")
        continue

    common.check_split(fold, train)          # wrong length? bad index? typo?
    val_preds, truth = fit_predict_lgbm(fold)
    # naming the model and the split means your score gets saved for later
    eval = common.score(truth, val_preds, MODEL, name,
                        note="LightGBM on RDKit descriptors")
    results[name] = eval
    preds_by_split[name] = (val_preds, truth, fold)
    print(f"{name:11s} val MA-RAE = {eval['RAE'].mean():.3f}   "
          f"({(fold=='val').sum()} molecules held out)")

In [ ]:
if not results:
    print("Nothing to compare yet -- implement a split above and re-run.")
else:
    comparison = pd.DataFrame({name: metrics["RAE"] for name, metrics in results.items()})
    fig, ax = plt.subplots(figsize=(10, 4.5))
    comparison.plot(kind="bar", ax=ax)
    ax.set_ylabel("RAE (lower is better)")
    ax.legend(); plt.tight_layout(); plt.show()
    display(comparison.round(3))

**Why does the split matter?**

**A validation split is a hypothesis about how the model will be used.** If you
will only ever predict compounds closely related to what you have measured, the
random split is honest. If you will predict next month's designs, it may overpromise and underdeliver.

Here, the test set is late-stage compounds from the same campaign. Which of
your four numbers do you now believe?

Why do you think some endpoints performed worse for some splits?

### Saving splits

You just built three splits and got a bunch of scores out. Those numbers are only
comparable to each other if every model you try today is scored on **the same
molecules**. To make this easier (and support the validation leaderboard!), we provide a way to save the splits now, so you can score all your models against them
for the rest of the day.

`save_split` stores the *labels*, not the recipe: a molecule-by-molecule
`"train"`/`"val"` assignment. You save as many as you like under names you
choose (and will remember!).
```python
common.save_split(fold, train, method)
# fold = the 'train'/'val' labels
# train = the dataset you generated the split for
# method = the name you give your split
```

Every time you train a new model, you can pick which validation split you want to use by name:
```python
fold, meta = common.load_split(train, name="your-split-name")
```


In [ ]:
for name in results:
    _, _, fold = preds_by_split[name]
    common.save_split(fold, train, method=name)

common.list_splits()

### Score every model against the same few splits

From here on, we recommend scoring each model
against **a few** splits and logging all of them:

```python
common.score(truth, preds, "my-model", "temporal")
```

That builds a `scores.csv` in your folder that saves across the models and splits you call `common.score` on.

**Keep at least one split fixed for the whole day.** If model A was scored on
  temporal + custom split and model B on random + similarity, it's difficult to compare the models. One shared column is what makes rows comparable.


In [ ]:
# Example of logging scores for lgbm-rdkit
EVAL_SPLITS = common.check_eval_splits([n for n in MY_SPLITS if n in results])

MODEL = "lgbm-rdkit"          # the name this model goes into the table under

for name in EVAL_SPLITS:
    fold, _ = common.load_split(train, name=name, verbose=False)
    val_preds, truth = fit_predict_lgbm(fold)
    # naming the model and the split is what puts it in the table;
    # if you plan to make a lot of models today, the note is worth filling in
    common.score(truth, val_preds, MODEL, name, note="LightGBM on RDKit descriptors")

common.score_matrix()

---
## 3. (Optional) Cross-validation

Everything so far rested on **one** held-out set. This is a fine validation approach, and is often used when you have a lot of data, a big model, and a lot of computational cost associated with training a model on multiple splits.

But, if you're not in that regime, cross-validation is a commonly used approach. Cross-validation runs the same split over *k* different folds and reports the
spread. It's possible that some folds will be harder than others -- what does that indicate about your data and your model?

> Each fold is a normal split &mdash; a `"train"`/`"val"` label per row. A
> k-fold scheme is just a **list of k of them**, where every molecule is in
> `val` exactly once.

**This is the expensive section.** 5 folds &times; 9 endpoints is 45 model fits.
The cell below trains on a single endpoint of your choosing, which is enough to see the spread; test other `CV_ENDPOINTS` when you have time to spare.


In [ ]:
# a helper function to run CV on a list of folds and report mean +/- spread.
def run_cv(folds, label, endpoints=None, model=None):
    """Score a list of folds and report mean +/- spread.

    Pass `model` to log the scheme's mean as one row in your score table.
    """
    scores = []
    for i, fold in enumerate(folds):
        common.check_split(fold, train)
        val_preds, truth = fit_predict_lgbm(fold, endpoints=endpoints)
        ma_rae = common.score(truth, val_preds, endpoints=endpoints)["RAE"].mean()
        scores.append(ma_rae)
        print(f"  fold {i}: MA-RAE = {ma_rae:.3f}   "
              f"({(fold == 'val').sum()} val, {(fold == 'train').sum()} train)")
    scores = np.asarray(scores)
    print(f"{label}: {scores.mean():.3f} +/- {scores.std():.3f} over {len(scores)} folds")

    if model:
        common.log_score(model, label, scores.mean(),
                         note=f"mean of {len(scores)} folds, sd {scores.std():.3f}")
    return scores

Let's start with a random k-fold -- same idea as random_split, but instead of cutting once we deal every molecule into one of k piles.

In [ ]:
def kfold_random(df, k, seed=0):
    random.seed(seed) # set seed for reproducibility
    indices = df.index.tolist() # create a list of indices
    random.shuffle(indices) # shuffle once, then deal into piles

    # cut the shuffled list into k blocks.
    fold_size = len(indices) // k
    piles = []
    for i in range(k):
        ### TODO ###
        # where does pile i start, and where does it end?
        start = ...
        end = ...
        ### END TODO ###
        if i == k - 1:
            end = len(indices)   # last pile mops up whatever did not divide evenly
        piles.append(indices[start:end])

    # each pile takes a turn as the validation set
    folds = []
    for pile in piles:
        val_indices = set(pile)
        labels = []
        for i in df.index:
            if i in val_indices:
                labels.append("val")
            else:
                labels.append("train")
        folds.append(pd.Series(labels, index=df.index))
    return folds

In [ ]:
# now let's actually run CV!
CV_ENDPOINTS = [common.ENDPOINTS[0]] # pick which endpoint you want to look at!
print(common.ENDPOINTS[0])
K = 5
cv_random = run_cv(kfold_random(train, K), "random 5-fold", endpoints=CV_ENDPOINTS)

### &#9654;&#65039; Now one that holds out whole regions of chemical space

Random k-fold has the same flaw the random split had, and cross-validation does
not fix it: every fold has close analogues of its validation molecules sitting
in its own training set. Five folds of the same lie gives you a very stable
wrong answer &mdash; notice how tight that spread was.

**Leave-one-cluster-out** attacks that directly. Group the molecules by
structure first, then hold out one whole group per fold. Now a fold has to
predict a region of chemical space it has never seen, which is much closer to
what "next month's compounds" actually means.

Clusters will not come out the same size, so your folds will not either. That
is fine and worth looking at &mdash; a fold holding out one big tight series is
a genuinely different test from one holding out a scatter of oddities.

*Which do you expect: a higher mean than random k-fold, a wider spread, or both?*


In [ ]:
# Cluster the fingerprints, then hold out one cluster at a time.
# k-means over fingerprint bits is the quick version. Butina clustering on
# Tanimoto distance is the more chemically standard one -- try swapping it in!
from sklearn.cluster import KMeans

def kfold_cluster(df, k, seed=0):
    # same Morgan fingerprints as the similarity split above
    gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
    fps = []
    for smi in df["SMILES"]:
        mol = Chem.MolFromSmiles(smi)
        fps.append(gen.GetFingerprint(mol) if mol is not None else None)
    X = common.fingerprints_to_array(fps)   # k-means needs numbers, not bit vectors

    ### TODO ###
    # group the molecules into k clusters.
    # clusters[i] is the cluster that molecule i belongs to.
    clusters = KMeans(n_clusters=..., random_state=seed, n_init=10).fit_predict(...)
    ### END TODO ###

    print("cluster sizes:", np.bincount(clusters))

    # each cluster takes a turn as the validation set
    folds = []
    for col in range(k):
        labels = []
        for molecule_cluster in clusters:
            if molecule_cluster == col:
                labels.append("val")
            else:
                labels.append("train")
        folds.append(pd.Series(labels, index=df.index))
    return folds

In [ ]:
# run CV here too!
cv_cluster = run_cv(kfold_cluster(train, K), "leave-one-cluster-out", endpoints=CV_ENDPOINTS)

We haven't provided an implementation to use a CV scheme in the validation leaderboard -- but we're happy to brainstorm with you if you're interested in that idea!

---
## 4. The validation leaderboard!

Pick the split you trust most and take its score. That number is now your
**prediction of your own test performance** &mdash; and you will submit it with
your predictions, before you find out whether it was right.

When the test scores land there is a second leaderboard, ranking pairs on

$$|\text{your validation estimate} - \text{your actual test score}|$$

That doesn't reward *being* accurate &mdash; it rewards *knowing how accurate you are*. A
pair scoring MA-RAE 0.75 who predicted 0.75 beats a pair scoring 0.60 who
predicted 0.45. The second pair built a more accurate model, but has a worse estimate of what it
does on new chemistry.

> **Look at your library now.** Which saved split would you stake your
> calibration score on, and what would you have to believe about the test set
> for that to be the right call?
> **Remember:** the test set is split temporally -- the compounds synthesized last are in it.


---
### Save your (better) baseline model predictions!

Something to keep in mind:

> **You validate on a split. You fit the model you actually submit on
> everything.**

The split exists to *estimate* performance &mdash; you hold molecules out,
predict them, and compare. Once you have that estimate, the held-out molecules
have done their job, and throwing away a fifth of your labelled data to make
test predictions would just give you a worse model for no reason. So before you submit your
final test predictions, it may be wise to retrain your model on all of the training data.

Two consequences worth knowing:

- Your submitted model has seen ~25% more data than the one you validated, so
  it should be **slightly better** than your estimate says. Your validation
  number is a mild underestimate of your own performance &mdash; which, on the
  calibration leaderboard, biases you toward predicting too much error.
- Nothing above ever trains on `test`. The test labels are blinded; the only
  thing you learn about them is what comes back from the leaderboard.

In [ ]:
# No split here on purpose: this is the model you would submit, so it trains
# on every labelled molecule. The splits above were for estimating its score.
test = common.load_test()
X_test = common.rdkit_descriptors(test["SMILES"])
X_train, X_test = common.clean_features(X_all, X_test) # clean both the same way

template = LGBMRegressor(n_estimators=400, learning_rate=0.05,
                         num_leaves=31, verbose=-1, n_jobs=-1)

# a submission is just the molecule ids plus one column per endpoint
test_preds = pd.DataFrame({"Molecule Name": test["Molecule Name"]})

for endpoint in common.ENDPOINTS:
    y_train = train[endpoint]
    ok = y_train.notna().to_numpy()
    model = clone(template)      # a fresh copy per endpoint
    model.fit(X_train[ok], y_train[ok])
    test_preds[endpoint] = model.predict(X_test)

test_preds.head()

#### Save your predictions: write a submission file!
It does not use up your submissions/day until you drag it into the scoring folder -- so you should write submissions often! Saving predictions is helpful because multiple models' predictions can be *ensembled* together to yield better predictions.

`common.score_matrix()` will display all the models and splits you've tried so far, and can inform which split you want to use in your prepared submission.**If you do not like any of the numbers you have**, come up with a new split you like better!

In [ ]:
common.score_matrix()

In [ ]:
SPLIT = ...
common.prepare_submission(test_preds, MODEL, SPLIT,
                          why="LightGBM on RDKit descriptors, one model per endpoint")

This is the reference baseline &mdash; the same recipe OpenADMET used
as their own comparison model.

**Now, let's build some better models!**

---
## 5. (Optional) Go further

You're not stuck with splits we scaffolded! In the real challenge one of the top-5
finishers (*shin-chan*) used a **difficulty-based** split &mdash; holding out
the molecules the model itself found hardest &mdash; to force honest validation
against awkward chemistry. Another top-20 finisher used similarity.

**Single splits to try**

- **property-based** &mdash; hold out the most lipophilic decile, or the
  heaviest, or the most flexible. Asks: does this model work at the edge of the
  property range we care about?
- **sparsity-based** &mdash; hold out the compounds with the fewest measured
  endpoints. Those are the ones a real project has least data on.
- **difficulty-based** &mdash; fit once, hold out the compounds with the largest
  residuals, refit. Deliberately adversarial.
- **one series out** &mdash; pick a single large scaffold family and hold out
  all of it. Mimics "we are moving to a new series next quarter."

**CV schemes to try**

- **rolling origin** &mdash; the temporal version of k-fold. Cut into `k + 1`
  chronological blocks; fold *i* trains on blocks `0..i` and validates on block
  `i + 1`, with everything later labelled `"unused"` so no fold ever trains on
  its own future. Closest in spirit to the real train/test split.
- **repeated random k-fold** &mdash; run `kfold_random` with three seeds and
  look at the spread of the *means*. Tells you how much your 5-fold estimate
  itself wobbles.
- **grouped k-fold on scaffolds** &mdash; like leave-one-cluster-out, but the
  groups are Bemis-Murcko scaffolds rather than fingerprint clusters.
- **purged / embargoed** &mdash; leave a gap between train and val blocks.
  Compounds registered next to each other are usually batch-mates, so adjacent
  numbers leak.
- **nested** &mdash; tune hyperparameters inside each training fold, score on
  the outer val block, so your tuning never sees the validation molecules.

Templates for both are below. Anything you build that passes `check_split`
works with everything else in this notebook.


In [ ]:
# ---- a single split -------------------------------------------------------
# in : df, a DataFrame with len(df) rows
# out: a pd.Series of "train"/"val", same length, index=df.index

def my_split(df, frac_val=0.2):
    ### YOUR CODE ###
    raise NotImplementedError
    ### END YOUR CODE ###

fold = my_split(train)
common.check_split(fold, train)        # wrong length? misaligned index? typo?

preds, truth = fit_predict_lgbm(fold)
score = common.score(truth, preds)["RAE"].mean()
print(f"MA-RAE on your split: {score:.3f}")

# happy with it? put it in the library.
# common.save_split(fold, train, method="my split", name="CHANGE-ME",
#                   rationale="CHANGE-ME -- one sentence you would defend",
#                   val_score=float(score))


In [ ]:
# ---- a CV scheme ----------------------------------------------------------
# in : df
# out: a list of k Series. "unused" is allowed as a third label, for schemes
#      like rolling origin where a fold must ignore part of the data.

def my_cv(df, k):
    ### YOUR CODE ###
    raise NotImplementedError
    ### END YOUR CODE ###

scores = run_cv(my_cv(train, K), "my CV scheme", endpoint=CV_ENDPOINTS)

# happy with it? save every fold under one scheme name.
# SCHEME = "cv-CHANGE-ME"
# for i, fold in enumerate(my_cv(train)):
#     common.save_split(fold, train, method="CHANGE-ME",
#                       name=f"{SCHEME}-fold{i}",
#                       rationale="CHANGE-ME",
#                       val_score=float(scores[i]))

---
## 6. (Optional) Where should you *not* trust this model?
Why are we doing all this?

To have a usable model that helps people make decision, you need both accuracy and *also* knowledge about how confident those predictions are. One way to think about this is through an accurate validation that captures error on an unknown test set -- another way to think about this is through an "applicability domain". The applicability domain specifies where the model is likely to function well, and on which inputs it will probably not give you good predictions.


We walk through one way to assess applicability domain: take the predictions you already have, bucket the validation molecules by how
similar they are to the training set, and look at the error in each bucket.

In [ ]:
# Which split you look at this through changes the answer.
# Swap in any name from MY_SPLITS you want to inspect.
LOOK_AT = SPLIT
val_preds, truth, fold = preds_by_split[LOOK_AT]

# same fingerprints as your similarity split
_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)
fp_all = [_gen.GetFingerprint(Chem.MolFromSmiles(s)) if Chem.MolFromSmiles(s) else None
          for s in train["SMILES"]]
train_idx = np.where((fold == "train").to_numpy())[0]
val_idx = np.where((fold == "val").to_numpy())[0]

nn_similarity = common.nearest_neighbour_similarity(
    [fp_all[i] for i in val_idx], [fp_all[i] for i in train_idx])

merged = truth.merge(val_preds, on="Molecule Name", suffixes=("_true", "_pred"))
merged["nn_sim"] = nn_similarity
merged["bucket"] = pd.qcut(merged["nn_sim"], 4, duplicates="drop")

rows = []
for bucket, group in merged.groupby("bucket", observed=True):
    err = np.concatenate([(group[endpoint + "_true"] - group[endpoint + "_pred"])
                          .abs().dropna().to_numpy()
                          for endpoint in common.ENDPOINTS])
    rows.append({"similarity bucket": str(bucket), "n errors": len(err),
                 "mean |error|": err.mean()})
domain = pd.DataFrame(rows)
domain

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(range(len(domain)), domain["mean |error|"], color="#468")
ax.set_xticks(range(len(domain)))
ax.set_xticklabels(domain["similarity bucket"], rotation=20, fontsize=8)
ax.set_xlabel("Tanimoto similarity to nearest training molecule")
ax.set_ylabel("mean absolute error")
ax.set_title("Applicability domain: error vs. how familiar the molecule is")
plt.tight_layout(); plt.show()

Something like *"below Tanimoto 0.4 we
would not trust this model"* is an **applicability domain** &mdash; a decision
rule for when to believe your own predictions.

Is Tanimoto similarity of Morgan fingerprints the most relevant "difficulty" metric? Feel free to implement a better one! Answering the question of where your model is reliable is necessary to build a model that will be useful for important decisions.